# Optimize Model for Inference

**Previous:** [03-download-trained-model.ipynb](03-download-trained-model.ipynb) | **Next:** [05-evaluate-model.ipynb](05-evaluate-model.ipynb)

---

This notebook optimizes the fine-tuned model for faster inference and deployment.

## What This Notebook Does

1. **Merges LoRA Adapters**: Combines LoRA weights with base model for standalone deployment
2. **Quantizes Model**: Reduces model size using INT8/INT4 quantization
3. **Exports to ONNX**: Converts to ONNX format for optimized cross-platform inference
4. **Benchmarks Performance**: Measures latency and throughput improvements
5. **Validates Accuracy**: Ensures optimizations don't degrade model quality

## Why Optimize?

| Optimization | Benefit | Trade-off |
|-------------|---------|-----------|
| **LoRA Merge** | Simpler deployment, faster loading | Larger model file |
| **Quantization** | 50-75% smaller, 2-4x faster | Slight accuracy loss |
| **ONNX Export** | Hardware-optimized, portable | Limited to inference only |

## Optimization Options

Configured in `configs/optimization_config.yaml`:
- **Quantization**: INT8, INT4, or FP16
- **ONNX Export**: With or without optimization
- **Merge Strategy**: Full merge or keep adapters separate

## Prerequisites

- Trained model downloaded (notebook 03-download-trained-model.ipynb)
- Sufficient disk space for optimized models (~10-30GB)

## Expected Duration

~5-15 minutes depending on model size and optimizations selected

In [ ]:
# 1. Import Libraries
import json
import time
from pathlib import Path
from dataclasses import dataclass
import logging
from typing import List, Dict, Any

from src.training.model_optimizer import (
    quantize_model,
    export_to_onnx,
    summarize_artifact,
    benchmark_inference,
)


**Why these imports?** The `model_optimizer` module contains pre-built functions for quantization, ONNX export, benchmarking, and artifact summarization. We reuse these tested functions rather than writing optimization code from scratch.

In [ ]:
# 2. Load Optimization Config
config_path = Path("configs/optimization_config.yaml")
if not config_path.exists():
    raise FileNotFoundError(f"Missing config: {config_path}")

import yaml
with open(config_path) as f:
    raw_cfg = yaml.safe_load(f)
raw_cfg

**Why load config?** The optimization config specifies quantization method (int8/int4), ONNX opset version, benchmark parameters, and output paths. This keeps optimization settings separate from code.

In [ ]:
# 3. Quantize Model
base_model_id = raw_cfg["base_model_id"]
quant_cfg = raw_cfg["quantization"]
quant_dir = Path(raw_cfg["output_root"]) / f"{base_model_id.replace('/', '_')}-quant"

if quant_cfg.get("enabled", True):
    quant_result = quantize_model(
        base_model_id,
        quant_dir,
        method=quant_cfg.get("method", "int8"),
        dtype=quant_cfg.get("dtype"),
    )
    quant_meta = summarize_artifact(quant_result)
else:
    quant_result = None
    quant_meta = {"enabled": False}
quant_meta

**Why quantize?** Quantization converts float32 weights to lower precision (int8/int4), dramatically reducing model size and improving CPU inference speed with minimal accuracy loss. This is critical for embedded deployment.

In [ ]:
# 4. Export ONNX
onnx_cfg = raw_cfg["onnx"]
onnx_path = Path(raw_cfg["output_root"]) / f"{base_model_id.replace('/', '_')}.onnx"
if onnx_cfg.get("enabled", True):
    onnx_result = export_to_onnx(
        base_model_id,
        onnx_path,
        opset=onnx_cfg.get("opset", 17),
        sequence_length=onnx_cfg.get("sequence_length", 128),
    )
    onnx_meta = summarize_artifact(onnx_result)
else:
    onnx_meta = {"enabled": False}
onnx_meta

**Why ONNX?** ONNX Runtime provides cross-platform inference with hardware-specific optimizations (CPU vectorization, memory layout). It's faster than raw PyTorch on CPU and enables deployment on non-Python environments.

In [ ]:
# 5. Benchmark Quantized Artifact (if present)
if quant_result:
    bench_stats = benchmark_inference(quant_result.artifact_path, repetitions=raw_cfg["benchmark"]["repetitions"], max_new_tokens=raw_cfg["benchmark"]["max_new_tokens"], prompt=raw_cfg["benchmark"]["prompt"])
else:
    bench_stats = {"skipped": True}
bench_stats

**Why benchmark?** We measure actual inference latency on CPU hardware to validate that optimizations meet the <50ms target. Benchmarking reveals if further optimization or hardware upgrades are needed.

In [ ]:
# 6. Compare Sizes
sizes = {}
if quant_result:
    sizes["quantized_bytes"] = quant_result.size_bytes
if onnx_path.exists():
    sizes["onnx_bytes"] = onnx_path.stat().st_size
sizes

In [ ]:
# 7. Persist Summary
summary = {
    "quantization": quant_meta,
    "onnx": onnx_meta,
    "benchmark": bench_stats,
    "sizes": sizes,
}
summary_path = Path(raw_cfg["output_root"]) / "summary.json"
summary_path.parent.mkdir(parents=True, exist_ok=True)
summary_path.write_text(json.dumps(summary, indent=2))
summary_path, summary

## Summary

✅ **Model successfully optimized for inference!**

**What was created:**
- Merged model (LoRA adapters integrated)
- Quantized model (reduced size and faster inference)
- ONNX model (optional, optimized for deployment)

**Next steps:**
- Evaluate optimized model (notebook 05-evaluate-model.ipynb)
- Build container image (notebook 06-push-to-acr.ipynb)
- Deploy for inference (notebook 07-deploy-inference.ipynb)

---

## Navigation

**Previous:** [03-download-trained-model.ipynb](03-download-trained-model.ipynb) | **Next:** [05-evaluate-model.ipynb](05-evaluate-model.ipynb)